# test to extract data 

:::{tip}
* make the todo list hahah
* understand the data
:::

In [1]:
import pandas as pd
import numpy as np
import unicodedata
from IPython.display import display
from pathlib import Path
from collections import OrderedDict, defaultdict
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

In [2]:
# =========================
# 1) Load the 3 tables
# =========================

TIC_df = pd.read_csv("data/TECNOLOGÍAS DE INFORMACIÓN Y COMUNICACIÓN__sheet_1.csv", sep=",")
EQ1_df = pd.read_csv("data/EQUIPAMIENTO DEL HOGAR__sheet_1.csv", sep=",")
EQ2_df = pd.read_csv("data/EQUIPAMIENTO DEL HOGAR__sheet_2.csv", sep=",")
SB1_df = pd.read_csv("data/SERVICIOS BÁSICOS__sheet_1.csv", sep=",")
SB2_df = pd.read_csv("data/SERVICIOS BÁSICOS__sheet_2.csv", sep=",")
SB3_df = pd.read_csv("data/SERVICIOS BÁSICOS__sheet_3.csv", sep=",")
SB4_df = pd.read_csv("data/SERVICIOS BÁSICOS__sheet_12.csv", sep=",") 
VYH_df = pd.read_csv("data/VIVIENDA_Y_HOGAR__sheet_2.csv", sep=",") 


In [3]:
# =========================
# 2) Filter zone (Pando + Beni/Vaca Díez)
# =========================
def filter_zone(df):
    return df[
        # Tout Pando
        (df["DEPARTAMENTO"] == "Pando") |

        # Beni : provincia Vaca Diez
        ((df["DEPARTAMENTO"] == "Beni") & (df["PROVINCIA"] == "Vaca Diez")) |

        # Beni : municipios spécifiques
        ((df["DEPARTAMENTO"] == "Beni") & (df["MUNICIPIO/TIOC"].isin(["Reyes", "Santa Rosa", "Exaltación"]))) |

        # La Paz : municipio Ixiamas
        ((df["DEPARTAMENTO"] == "La Paz") & (df["MUNICIPIO/TIOC"] == "Ixiamas"))
    ].copy()

TIC_df = filter_zone(TIC_df)
EQ1_df = filter_zone(EQ1_df)
EQ2_df = filter_zone(EQ2_df)
SB1_df = filter_zone(SB1_df)
SB2_df = filter_zone(SB2_df)
SB3_df = filter_zone(SB3_df)
SB4_df = filter_zone(SB4_df)
VYH_df = filter_zone(VYH_df)


In [4]:
# =========================
# 3) Keep only TOTAL rows (drop Urbana/Rural rows)
#    then drop NIVEL and ÁREA
# =========================
def keep_only_totals(df):

    # remove Urbana / Rural
    df = df[df["ÁREA"] != "Urbana"]
    df = df[df["ÁREA"] != "Rural"]

    # drop useless columns
    if "NIVEL" in df.columns: df = df.drop(columns=["NIVEL"])
    if "ÁREA" in df.columns: df = df.drop(columns=["ÁREA"])

    
    # -------------------------
    # Create Amazonía Norte row
    # -------------------------

    # create empty row with same structure
    amazonia_row = df.iloc[0:1].copy()

    # set geographic labels
    amazonia_row["DEPARTAMENTO"] = "Amazonía Norte"
    if "PROVINCIA" in df.columns: amazonia_row["PROVINCIA"] = ""
    if "MUNICIPIO/TIOC" in df.columns: amazonia_row["MUNICIPIO/TIOC"] = ""

    # sum numeric columns
    df_municipalities = df.dropna(subset=["MUNICIPIO/TIOC"])
    numeric_cols = df_municipalities.select_dtypes(include=["number"]).columns
    for col in numeric_cols:
        amazonia_row[col] = df_municipalities[col].sum()

    # insert at the top
    df = pd.concat([amazonia_row, df], ignore_index=True)

    return df

TIC_df = keep_only_totals(TIC_df)
EQ1_df = keep_only_totals(EQ1_df)
EQ2_df = keep_only_totals(EQ2_df)
SB1_df = keep_only_totals(SB1_df)
SB2_df = keep_only_totals(SB2_df)
SB3_df = keep_only_totals(SB3_df)
SB4_df = keep_only_totals(SB4_df)
VYH_df = keep_only_totals(VYH_df)


In [5]:
# =========================
# 4) keep only the columns we want
# =========================

KEYS = ["DEPARTAMENTO", "PROVINCIA", "MUNICIPIO/TIOC"]
base_df = TIC_df[KEYS].copy()


# ==========
# A) EQ1
# ==========

eq1_keep = []
 
for year in ["2001", "2012", "2024"]:
    eq1_keep += [c for c in EQ1_df.columns if c.startswith(f"NÚMERO DE HOGARES | {year} | Total hogares")]
    eq1_keep += [c for c in EQ1_df.columns if c.startswith(f"NÚMERO DE HOGARES | {year} | Vehículo automotor")]
    eq1_keep += [c for c in EQ1_df.columns if c.startswith(f"NÚMERO DE HOGARES | {year} | Motocicleta o cuadratrack")]

# Remove duplicates while keeping order
eq1_keep = list(dict.fromkeys(eq1_keep))
EQ1_sel = EQ1_df[KEYS + eq1_keep].copy()

# Rename EQ2 prefix
rename_map_eq1 = {}

for c in EQ1_sel.columns:
    if c.startswith("NÚMERO DE HOGARES"):
        new = c.replace("NÚMERO DE HOGARES", "NÚMERO DE HOGARES CON VEHÍCULOS", 1)
        rename_map_eq1[c] = new

EQ1_sel = EQ1_sel.rename(columns=rename_map_eq1)


# ===========
# B) EQ2
# ===========

eq2_keep = [c for c in EQ2_df.columns if c.startswith("NÚMERO DE HOGARES")]
EQ2_sel = EQ2_df[KEYS + eq2_keep].copy()

# Rename EQ2 prefix
rename_map_eq2 = {}

for c in EQ2_sel.columns:
    if c.startswith("NÚMERO DE HOGARES"):
        new = c.replace("NÚMERO DE HOGARES", "NÚMERO DE HOGARES CON EQUIPAMIENTO DEL HOGAR | 2024", 1)
        rename_map_eq2[c] = new

EQ2_sel = EQ2_sel.rename(columns=rename_map_eq2)


# ===========
# C) TIC
# ===========

tic_keep = []

# 2001 variables
tic_keep += [c for c in TIC_df.columns if c.startswith("TOTAL HOGARES(1) | 2001 | Total")]
tic_keep += [c for c in TIC_df.columns if c.startswith("TOTAL HOGARES(1) | 2001 | Radio o equipo de sonido")]
tic_keep += [c for c in TIC_df.columns if c.startswith("TOTAL HOGARES(1) | 2001 | Televisor")]
tic_keep += [c for c in TIC_df.columns if c.startswith("TOTAL HOGARES(1) | 2001 | Teléfono o celular")]

# 2012 variables 
tic_keep += [c for c in TIC_df.columns if c.startswith("TOTAL HOGARES(1) | 2012 | Total")]
tic_keep += [c for c in TIC_df.columns if c.startswith("TOTAL HOGARES(1) | 2012 | Radio")]
tic_keep += [c for c in TIC_df.columns if c.startswith("TOTAL HOGARES(1) | 2012 | Televisor")]
tic_keep += [c for c in TIC_df.columns if c.startswith("TOTAL HOGARES(1) | 2012 | Computadora")]
tic_keep += [c for c in TIC_df.columns if c.startswith("TOTAL HOGARES(1) | 2012 | Servicio de Internet")]
tic_keep += [c for c in TIC_df.columns if c.startswith("TOTAL HOGARES(1) | 2012 | Servicio de telefonía fija o celular")]

# 2024 variables
tic_keep += [c for c in TIC_df.columns if c.startswith("TOTAL HOGARES(1) | 2024(2) | Total")]
tic_keep += [c for c in TIC_df.columns if c.startswith("TOTAL HOGARES(1) | 2024(2) | Radio o equipo de sonido")]
tic_keep += [c for c in TIC_df.columns if c.startswith("TOTAL HOGARES(1) | 2024(2) | Televisor")]
tic_keep += [c for c in TIC_df.columns if c.startswith("TOTAL HOGARES(1) | 2024(2) | Computadora o laptop o tablet")]
tic_keep += [c for c in TIC_df.columns if c.startswith("TOTAL HOGARES(1) | 2024(2) | Teléfono celular")]
tic_keep += [c for c in TIC_df.columns if c.startswith("TOTAL HOGARES(1) | 2024(2) | Internet fijo en la vivienda")]

tic_keep = list(dict.fromkeys(tic_keep))
TIC_sel = TIC_df[KEYS + tic_keep].copy()

# Rename TIC column labels exactly as requested
rename_map = {}
for c in TIC_sel.columns:
    new = c
    new = new.replace("TOTAL HOGARES(1)", "NÚMERO DE HOGARES CON TECNOLOGÍAS TIC") # remove "(1)" in the prefix
    new = new.replace("| 2024(2) |", "| 2024 |") # unify year label 2024(2) -> 2024
    new = new.replace("Teléfono o celular", "Teléfono") 
    new = new.replace("| 2012 | Radio", "| 2012 | Radio o equipo de sonido")
    new = new.replace("| 2012 | Computadora", "| 2012 | Computadora o laptop o tablet") 
    new = new.replace("| 2012 | Servicio de Internet", "| 2012 | Internet fijo en la vivienda") 
    new = new.replace("| 2012 | Servicio de telefonía fija o celular", "| 2012 | Teléfono")
    new = new.replace("| 2024 | Teléfono celular", "| 2024 | Teléfono")
    rename_map[c] = new

TIC_sel = TIC_sel.rename(columns=rename_map)


# ===========
# D) SB1
# ===========

sb1_keep = [ c for c in SB1_df.columns if c.startswith("NÚMERO DE VIVIENDAS | 2001")]
SB1_sel = SB1_df[KEYS + sb1_keep].copy()  

# Rename SB1
rename_map_sb1 = {}

for c in SB1_sel.columns:
    if c.startswith("NÚMERO DE VIVIENDAS | 2001"):
        new = c
        new = new.replace("NÚMERO DE VIVIENDAS", "NÚMERO DE VIVIENDAS POR FUENTE DE ELECTRICIDAD", 1)
        new = new.replace("| Total", "| Electricidad total", 1)
        new = new.replace("| Tiene", "| Tiene eléctrica", 1)
        new = new.replace("| No tiene", "| No tiene eléctrica", 1)
        rename_map_sb1[c] = new

SB1_sel = SB1_sel.rename(columns=rename_map_sb1)


# ===========
# E) SB2
# ===========

sb2_keep = [c for c in SB2_df.columns if c.startswith("NÚMERO DE PERSONAS(1)")]
SB2_sel = SB2_df[KEYS + sb2_keep].copy()

# Rename SB2 total columns
rename_map_sb2 = {}

for c in SB2_sel.columns:
    if c.startswith("NÚMERO DE PERSONAS(1)"):
        new = c
        new = new.replace("NÚMERO DE PERSONAS(1)", "NÚMERO DE PERSONAS POR FUENTE DE ELECTRICIDAD", 1)
        rename_map_sb2[c] = new

SB2_sel = SB2_sel.rename(columns=rename_map_sb2)


# ===========
# F) SB3
# ===========

sb3_keep = [c for c in SB3_df.columns if c.startswith("NÚMERO DE VIVIENDAS")]
SB3_sel = SB3_df[KEYS + sb3_keep].copy()

# Rename SB3 total columns
rename_map_sb3 = {}

for c in SB3_sel.columns:
    if c.startswith("NÚMERO DE VIVIENDAS"):
        new = c
        new = new.replace("NÚMERO DE VIVIENDAS", "NÚMERO DE VIVIENDAS POR FUENTE DE ELECTRICIDAD", 1)
        rename_map_sb3[c] = new

SB3_sel = SB3_sel.rename(columns=rename_map_sb3)



# ===========
# G) SB4
# ===========

sb4_keep = [c for c in SB4_df.columns if c.startswith("NÚMERO DE VIVIENDAS")]
SB4_sel = SB4_df[KEYS + sb4_keep].copy()

# Rename SB4 total columns
rename_map_sb4 = {}

for c in SB4_sel.columns:
    if c.startswith("NÚMERO DE VIVIENDAS"):
        new = c
        new = new.replace("NÚMERO DE VIVIENDAS", "NÚMERO DE VIVIENDAS SEGÚN ENERGÍA UTILIZADA PARA COCINAR", 1)
        rename_map_sb4[c] = new

SB4_sel = SB4_sel.rename(columns=rename_map_sb4)


# ===========
# H) VYH
# ===========

vyh_keep = []
vyh_keep += [c for c in VYH_df.columns if c.startswith("NÚMERO DE VIVIENDAS | Hotel, hostal, residencial, alojamiento")]
vyh_keep += [c for c in VYH_df.columns if c.startswith("NÚMERO DE VIVIENDAS | Hospital o clínica con internación")]

vyh_keep = list(dict.fromkeys(vyh_keep))
VYH_sel = VYH_df[KEYS + vyh_keep].copy()

# Rename VYH total columns
rename_map_vyh = {}

for c in VYH_sel.columns:
    if c.startswith("NÚMERO DE VIVIENDAS"):
        new = c
        new = new.replace("NÚMERO DE VIVIENDAS", "NÚMERO DE VIVIENDAS | 2024", 1)
        rename_map_vyh[c] = new

VYH_sel = VYH_sel.rename(columns=rename_map_vyh)


In [6]:
# ============================================================
# 5) Merge everything into ONE big wide table
# ============================================================

final_df = base_df.merge(SB1_sel, on=KEYS, how="left")
final_df = final_df.merge(SB3_sel, on=KEYS, how="left")
final_df = final_df.merge(SB2_sel, on=KEYS, how="left")
final_df = final_df.merge(SB4_sel, on=KEYS, how="left")
final_df = final_df.merge(EQ1_sel, on=KEYS, how="left")
final_df = final_df.merge(EQ2_sel, on=KEYS, how="left")
final_df = final_df.merge(TIC_sel, on=KEYS, how="left")
final_df = final_df.merge(VYH_sel, on=KEYS, how="left")

# Drop duplicated KEY columns created by merges (if any)
final_df = final_df.loc[:, ~final_df.columns.duplicated()].copy()

# Recreate N°
final_df = final_df.reset_index(drop=True)
final_df.insert(0, "N°", range(1, len(final_df) + 1))

# (save only when CSV1 is finished -> you asked for that)
final_df.to_csv("output/CSV_final.csv", index=False)


In [ ]:
# =========================
# 5b) Add SUPERFICIE (km²)
# =========================

def normalize_str(s):
    if pd.isna(s) or str(s).strip() == "":
        return ""
    s = unicodedata.normalize("NFD", str(s))
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return s.strip().lower()

area_source = pd.read_excel("data/municipality_areas.xlsx")
area_source["dept_key"] = area_source["Department"].apply(normalize_str)
area_source["muni_key"] = area_source["Municipality"].apply(normalize_str)

# Primary key: (department, municipality) to handle duplicate municipality names across departments
area_dict = {(r["dept_key"], r["muni_key"]): r["Area_km2"] for _, r in area_source.iterrows()}
# Fallback key: municipality name only (for cases with no ambiguity)
area_dict_muni = {r["muni_key"]: r["Area_km2"] for _, r in area_source.iterrows()}

def is_empty(val):
    return pd.isna(val) or str(val).strip() == ""

def lookup_area(dept, muni):
    dk, mk = normalize_str(dept), normalize_str(muni)
    return area_dict.get((dk, mk), area_dict_muni.get(mk))

# Municipality-level areas (direct lookup by department + name)
final_df["SUPERFICIE (km²)"] = None
muni_mask = ~final_df["MUNICIPIO/TIOC"].apply(is_empty)
for idx in final_df[muni_mask].index:
    final_df.at[idx, "SUPERFICIE (km²)"] = lookup_area(
        final_df.at[idx, "DEPARTAMENTO"], final_df.at[idx, "MUNICIPIO/TIOC"]
    )

# Province aggregate areas (sum of municipalities in that province)
prov_agg_mask = final_df["MUNICIPIO/TIOC"].apply(is_empty) & ~final_df["PROVINCIA"].apply(is_empty)
for idx, row in final_df[prov_agg_mask].iterrows():
    sub_mask = muni_mask & (final_df["DEPARTAMENTO"] == row["DEPARTAMENTO"]) & (final_df["PROVINCIA"] == row["PROVINCIA"])
    final_df.at[idx, "SUPERFICIE (km²)"] = final_df.loc[sub_mask, "SUPERFICIE (km²)"].sum()

# Department aggregate areas (sum of municipalities in that department)
dept_agg_mask = (
    final_df["MUNICIPIO/TIOC"].apply(is_empty) &
    final_df["PROVINCIA"].apply(is_empty) &
    (final_df["DEPARTAMENTO"] != "Amazonía Norte")
)
for idx, row in final_df[dept_agg_mask].iterrows():
    sub_mask = muni_mask & (final_df["DEPARTAMENTO"] == row["DEPARTAMENTO"])
    final_df.at[idx, "SUPERFICIE (km²)"] = final_df.loc[sub_mask, "SUPERFICIE (km²)"].sum()

# Amazonía Norte total
amazonia_idx = final_df[final_df["DEPARTAMENTO"] == "Amazonía Norte"].index[0]
final_df.at[amazonia_idx, "SUPERFICIE (km²)"] = final_df.loc[muni_mask, "SUPERFICIE (km²)"].sum()

# Place SUPERFICIE (km²) right after MUNICIPIO/TIOC
cols = list(final_df.columns)
cols.remove("SUPERFICIE (km²)")
cols.insert(cols.index("MUNICIPIO/TIOC") + 1, "SUPERFICIE (km²)")
final_df = final_df[cols]

# Re-save CSV with the area column included
final_df.to_csv("output/CSV_final.csv", index=False)

In [7]:
# ============================================================
# 6) transforme in a visual excel (thanks chatgpt)
# ============================================================

df = final_df.copy() 

def export_stair_excel(df, filename="output/CSV_final_in_excel.xlsx", header_rows=4):
    # ---------------------------
    # 1) Split columns
    # ---------------------------
    left_cols = [c for c in df.columns if " | " not in c]
    hier_cols = [c for c in df.columns if " | " in c]

    # ---------------------------
    # 2) Parse hierarchical columns and preserve original order
    # ---------------------------
    # Map (block, year, var, status) -> original column name (first occurrence wins)
    tuple_to_col = OrderedDict()
    for col in hier_cols:
        parts = [p.strip() for p in col.split(" | ")]
        parts += [""] * (4 - len(parts))  # pad to 4
        key = tuple(parts[:4])            # (block, year, var, status)
        tuple_to_col.setdefault(key, col)

    # Build ordered groups: blocks -> years -> vars -> statuses (all order-preserving)
    years_by_block = OrderedDict()
    vars_by_by = OrderedDict()
    status_by_byv = OrderedDict()

    for (b, y, v, s) in tuple_to_col.keys():
        years_by_block.setdefault(b, [])
        if y not in years_by_block[b]:
            years_by_block[b].append(y)

        vars_by_by.setdefault((b, y), [])
        if v not in vars_by_by[(b, y)]:
            vars_by_by[(b, y)].append(v)

        status_by_byv.setdefault((b, y, v), [])
        if s not in status_by_byv[(b, y, v)]:
            status_by_byv[(b, y, v)].append(s)

    # Final ordered hierarchical columns
    ordered_hier = []
    for b, years in years_by_block.items():
        for y in years:
            for v in vars_by_by[(b, y)]:
                for s in status_by_byv[(b, y, v)]:
                    ordered_hier.append(tuple_to_col[(b, y, v, s)])

    df2 = df[left_cols + ordered_hier].copy()

    # ---------------------------
    # 3) Workbook + styles
    # ---------------------------
    wb = Workbook()
    ws = wb.active
    ws.title = "data"

    header_fill = PatternFill(start_color="5F8FA6", end_color="5F8FA6", fill_type="solid")
    header_font = Font(bold=True, color="FFFFFF")
    header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
    thin = Side(style="thin", color="2F2F2F")
    header_border = Border(left=thin, right=thin, top=thin, bottom=thin)

    def style_range(r1, c1, r2, c2):
        """Apply header style + borders to a rectangular cell range."""
        for r in range(r1, r2 + 1):
            for c in range(c1, c2 + 1):
                cell = ws.cell(row=r, column=c)
                cell.fill = header_fill
                cell.font = header_font
                cell.alignment = header_align
                cell.border = header_border

    def merge_header(row, c1, c2, value):
        """Merge cells in a header row and style the whole merged range."""
        if c2 > c1:
            ws.merge_cells(start_row=row, start_column=c1, end_row=row, end_column=c2)
        ws.cell(row=row, column=c1, value=value)
        style_range(row, c1, row, c2)

    # ---------------------------
    # 4) Left (ID) headers merged vertically
    # ---------------------------
    for j, name in enumerate(left_cols, start=1):
        ws.merge_cells(start_row=1, start_column=j, end_row=header_rows, end_column=j)
        ws.cell(row=1, column=j, value=name)
        style_range(1, j, header_rows, j)

    # ---------------------------
    # 5) Hierarchical headers (4 rows)
    # ---------------------------
    start_col = len(left_cols) + 1
    col_ptr = start_col

    for b, years in years_by_block.items():
        b_start = col_ptr

        for y in years:
            y_start = col_ptr

            for v in vars_by_by[(b, y)]:
                v_start = col_ptr

                statuses = status_by_byv[(b, y, v)]
                for s in statuses:
                    ws.cell(row=4, column=col_ptr, value=s)  # status row
                    style_range(4, col_ptr, 4, col_ptr)
                    col_ptr += 1

                v_end = col_ptr - 1
                merge_header(3, v_start, v_end, v)  # variable row

            y_end = col_ptr - 1
            merge_header(2, y_start, y_end, y)      # year row

        b_end = col_ptr - 1
        merge_header(1, b_start, b_end, b)          # block row

    # ---------------------------
    # 6) Write data (below headers)
    # ---------------------------
    data_start_row = header_rows + 1
    for i, row in enumerate(df2.itertuples(index=False), start=data_start_row):
        for j, val in enumerate(row, start=1):
            ws.cell(row=i, column=j, value=val)

    # ---------------------------
    # 7) Column widths + header row heights
    # ---------------------------
    for j, name in enumerate(df2.columns, start=1):
        col_letter = get_column_letter(j)
        if name == "N°":
            ws.column_dimensions[col_letter].width = 6
        elif name in ("DEPARTAMENTO", "PROVINCIA", "MUNICIPIO/TIOC"):
            ws.column_dimensions[col_letter].width = 18
        else:
            ws.column_dimensions[col_letter].width = 20

    ws.row_dimensions[1].height = 22
    ws.row_dimensions[2].height = 22
    ws.row_dimensions[3].height = 38
    ws.row_dimensions[4].height = 18

    wb.save(filename)

export_stair_excel(final_df, "output/CSV_final_in_excel.xlsx")